In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
cols = ['unit', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

def load_data(file_path):
    df = pd.read_csv(file_path, sep=' ', header=None, index_col=False)
    df = df.iloc[:, :26]
    df.columns = cols
    return df

In [3]:
train_df = load_data('train_FD001.txt')

In [4]:
max_cycles = train_df.groupby('unit')['cycle'].max().reset_index()
train_df = train_df.merge(max_cycles, on='unit', suffixes=('', '_max'))
train_df['RUL'] = train_df['cycle_max'] - train_df['cycle']

# The Evaluation Logic

In [ ]:
def verify_outputs(predictions):
    """
    Logic to verify the correctness of analytical outputs.
    """
    checks = {
        "non_negative": np.all(predictions >= 0),
        "not_nan": not np.any(np.isnan(predictions)),
        "reasonable_range": np.all(predictions < 500) # Engines rarely last >500 cycles
    }
    
    for check, passed in checks.items():
        status = "PASSED" if passed else "FAILED"
        print(f"Validation Check [{check}]: {status}")
    
    return all(checks.values())



# Feature selection (using a few sensors for this example)


In [6]:
features = ['cycle', 's2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
X = train_df[features]
y = train_df['RUL']

# Model Comparison & Benchmarking

Train Model A: Linear Regression (Your previous specialty)


In [7]:
lr_model = LinearRegression()
lr_model.fit(X, y)
lr_preds = lr_model.predict(X)

Train Model B: Random Forest (To compare performance)


In [8]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)
rf_preds = rf_model.predict(X)

Comparative Evaluation


In [11]:
print("--- Linear Regression Evaluation ---")
verify_outputs(lr_preds)
print(f"R2 Score: {r2_score(y, lr_preds):.4f}")

--- Linear Regression Evaluation ---
Validation Check [non_negative]: FAILED
Validation Check [not_nan]: PASSED
Validation Check [reasonable_range]: PASSED
R2 Score: 0.6554


# Why it failed


Linear Regression is a simple mathematical line. If the data trends downward steeply, the model will just keep following that line below zero, even if "zero" should be the absolute limit for a failing engine.

# How to fix the code


To ensure correctness I will clip the predictions so they never fall below zero.

In [13]:
lr_preds = np.maximum(0, lr_model.predict(X))

Now i will run the validation again


In [15]:
print("--- Linear Regression Evaluation (Fixed) ---")
verify_outputs(lr_preds)
print(f"R2 Score: {r2_score(y, lr_preds):.4f}")

--- Linear Regression Evaluation (Fixed) ---
Validation Check [non_negative]: PASSED
Validation Check [not_nan]: PASSED
Validation Check [reasonable_range]: PASSED
R2 Score: 0.6661


In [16]:
from sklearn.metrics import mean_absolute_error
print(f"LR MAE: {mean_absolute_error(y, lr_preds)}")
print(f"RF MAE: {mean_absolute_error(y, rf_preds)}")

LR MAE: 30.00580734605589
RF MAE: 9.39394406475692


# Load the test features

In [26]:
test_df = load_data('test_FD001.txt')

Load the actual RUL values (Ground Truth)

In [27]:
truth_df = pd.read_csv('RUL_FD001.txt', header=None, names=['RUL_actual'])
truth_df['unit'] = truth_df.index + 1  # Assigning unit IDs to match test_df

Select the same features used during training

In [28]:
X_test = test_df.groupby('unit').last()[features]

Predict using Linear Regression with my 'clipping' fix

In [29]:
lr_test_preds = np.maximum(0, lr_model.predict(X_test))

Run the validation check 

In [30]:
print("--- Test Set Validation ---")
is_valid = verify_outputs(lr_test_preds)

--- Test Set Validation ---
Validation Check [non_negative]: PASSED
Validation Check [not_nan]: PASSED
Validation Check [reasonable_range]: PASSED


Calculate final performance metrics

In [31]:
if is_valid:
    test_score = r2_score(truth_df['RUL_actual'], lr_test_preds)
    test_mae = mean_absolute_error(truth_df['RUL_actual'], lr_test_preds)
    
    print(f"Test R2 Score: {test_score:.4f}")
    print(f"Test Mean Absolute Error: {test_mae:.2f} cycles")

Test R2 Score: 0.4492
Test Mean Absolute Error: 25.58 cycles


Predict using the Random Forest agent

In [21]:
rf_test_preds = rf_model.predict(X_test)

Run validation on the second agent

In [23]:
print("--- Random Forest Test Validation ---")
verify_outputs(rf_test_preds)

--- Random Forest Test Validation ---
Validation Check [non_negative]: PASSED
Validation Check [not_nan]: PASSED
Validation Check [reasonable_range]: PASSED


True

Benchmarking Agent B vs Agent A

In [24]:
rf_test_score = r2_score(truth_df['RUL_actual'], rf_test_preds)
rf_test_mae = mean_absolute_error(truth_df['RUL_actual'], rf_test_preds)

In [25]:
print(f"RF Test R2 Score: {rf_test_score:.4f}")
print(f"RF Test MAE: {rf_test_mae:.2f} cycles")

RF Test R2 Score: 0.5789
RF Test MAE: 20.26 cycles
